In [1]:
# bibliotecques
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Chemin d'accès 
ROOT = Path.cwd().parent 
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print(f"Project root: {ROOT}")

from src.data_loader import *
from src.periodicity import *
from src.hull_white import *
from src.sinusoidal_hw import *
from src.monte_carlo import *
from src.calibration import *
print("Imports OK")

Project root: /Users/badaire/Desktop/Cursus_mathematiques/M2QF/MOI/PROJETS/sinusoidal_hw
Imports OK


In [2]:
# 1. Chargement des données
data_path = ROOT / 'data' / 'fredgraph.csv'
loader = YieldDataLoader(str(data_path))
df = loader.load_data()

Données chargées. 1292 lignes supprimées (NaN/Jours fériés).
Période: 1993-10-01 à 2022-12-30


In [3]:
# 2. Sélection de la courbe de taux à calibrer
# Le papier mentionne la période finissant en Dec 2022. 
# Nous calibrons sur la DERNIÈRE date disponible pour avoir l'état "actuel" du marché.

calibration_date = df.index[-1]
current_curve = df.loc[calibration_date]

print(f"--- Calibration sur la courbe du {calibration_date.date()} ---")
print(current_curve)

# Préparation des inputs
# Mapping des maturités (ex: '1Y' -> 1.0)
maturities_map = {'1Y': 1.0, '2Y': 2.0, '3Y': 3.0, '5Y': 5.0, '7Y': 7.0, '10Y': 10.0, '20Y': 20.0, '30Y': 30.0}
maturities = []
yields = []

for tenor, years in maturities_map.items():
    if tenor in current_curve:
        maturities.append(years)
        yields.append(current_curve[tenor]) # Déjà en décimales via data_loader

r0 = yields[0] # Le taux court initial est approximé par le taux 1 an (1.22% dans le papier)
print(f"Taux court initial (r0) : {r0:.4%}")


--- Calibration sur la courbe du 2022-12-30 ---
1Y     0.0473
2Y     0.0441
3Y     0.0422
5Y     0.0399
7Y     0.0396
10Y    0.0388
20Y    0.0414
30Y    0.0397
Name: 2022-12-30 00:00:00, dtype: float64
Taux court initial (r0) : 4.7300%


In [4]:
# 3. Initialisation du Calibrateur
calibrator = Calibrator(maturities, yields)

In [5]:
# 4. Calibration Hull-White Standard
res_hw = calibrator.calibrate_standard_hw(r0)
print("\n--- Résultats Calibration HW Standard ---")
print(res_hw)

Début calibration Hull-White Standard (Analytique)...

--- Résultats Calibration HW Standard ---
{'kappa': np.float64(1.857874328663124), 'theta': np.float64(0.03985419377184708), 'sigma': np.float64(0.001), 'rmse': np.float64(0.006259978763175738), 'success': True}


In [6]:
# 5. Calibration Sinusoidal HW
# IMPORTANT : On utilise le OMEGA calculé en Phase 2
# Remplacez cette valeur par celle trouvée dans le notebook 2 si différente

FINAL_OMEGA_FROM_PHASE_2 = 0.000859
res_sin = calibrator.calibrate_sinusoidal_hw(r0, fixed_omega=FINAL_OMEGA_FROM_PHASE_2)
print("\n--- Résultats Calibration Sinusoidal HW ---")
print(res_sin)

Début calibration Sinusoidal HW (Monte Carlo, omega=0.000859)...
Cela peut prendre quelques minutes...

--- Résultats Calibration Sinusoidal HW ---
{'kappa_0': np.float64(0.2746165071590746), 'A': np.float64(0.19525189858912492), 'omega': 0.000859, 'theta': np.float64(0.03893298807491556), 'sigma': np.float64(0.009751499540932539), 'rmse': np.float64(0.008323237027967677), 'success': False}


In [7]:
# 6. Comparaison RMSE
print("\n--- Comparaison RMSE ---")
print(f"Standard HW   : {res_hw['rmse']:.6f}")
print(f"Sinusoidal HW : {res_sin['rmse']:.6f}")


--- Comparaison RMSE ---
Standard HW   : 0.006260
Sinusoidal HW : 0.008323
